In [1]:
import pandas as pd
import numpy as np
import requests
from tqdm import tqdm
import time
import os

# تحميل الداتا
data = pd.read_csv('Applications.csv')  # استبدلي باسم ملف الداتا بتاعك

# استخراج النقاط الفريدة (LAT, LON)
unique_points = data[['LAT', 'LON']].drop_duplicates().reset_index(drop=True)
print(f"عدد النقاط الفريدة: {len(unique_points)}")

# التحقق من وجود ملف التخزين المؤقت
cache_file = 'unique_points_elevations.csv'
if os.path.exists(cache_file):
    print("تحميل الارتفاعات من ملف التخزين المؤقت...")
    cached_elevations = pd.read_csv(cache_file)
    # التأكد من أن جميع النقاط الفريدة موجودة في الملف
    unique_points = unique_points.merge(
        cached_elevations[['LAT', 'LON', 'elevation']],
        on=['LAT', 'LON'],
        how='left'
    )
else:
    unique_points['elevation'] = np.nan

# دالة لجلب الارتفاع من Open Elevation API
def get_elevation(lat, lon, max_attempts=10, step=0.0001):
    """
    جلب الارتفاع لنقطة (lat, lon) مع محاولات متعددة في اتجاهات مختلفة.
    لو فشل، يتم إرجاع متوسط الارتفاعات للنقاط القريبة.
    """
    base_url = "https://api.open-elevation.com/api/v1/lookup"
    
    # تركيبات مختلفة لتغيير الإحداثيات
    offsets = [
        (0, 0),  # النقطة الأصلية
        (step, 0), (-step, 0), (0, step), (0, -step),  # تغييرات في محور واحد
        (step, step), (step, -step), (-step, step), (-step, -step),  # تغييرات في المحورين
        (step * 2, 0), (-step * 2, 0)  # تغييرات أكبر
    ]
    
    for attempt in range(min(max_attempts, len(offsets))):
        try:
            # حساب الإحداثيات الجديدة
            lat_offset, lon_offset = offsets[attempt]
            new_lat = lat + lat_offset
            new_lon = lon + lon_offset
            
            # إرسال الطلب
            locations = f"{new_lat},{new_lon}"
            response = requests.get(f"{base_url}?locations={locations}", timeout=10)
            
            if response.status_code == 200:
                result = response.json()
                if result['results'] and 'elevation' in result['results'][0]:
                    return result['results'][0]['elevation']
            # لو فيه خطأ في الاستجابة، ننتظر ونجرب الإحداثيات التالية
            time.sleep(0.5)
        except Exception as e:
            print(f"خطأ عند جلب الارتفاع لـ ({new_lat}, {new_lon}): {e}")
            time.sleep(0.5)
    
    # لو فشلنا بعد كل المحاولات، نحسب متوسط الارتفاعات للنقاط القريبة
    print(f"محاولة جلب ارتفاع بديل لـ ({lat}, {lon})")
    nearby_elevation = get_nearby_elevation(lat, lon, unique_points)
    if not np.isnan(nearby_elevation):
        return nearby_elevation
    
    # لو لسة مفيش ارتفاع، نرجع NaN
    print(f"تحذير: لم يتم العثور على ارتفاع لـ ({lat}, {lon})")
    return np.nan

# دالة لجلب متوسط الارتفاعات للنقاط القريبة
def get_nearby_elevation(lat, lon, points, max_distance=0.1):
    """
    حساب متوسط الارتفاعات للنقاط في نطاق max_distance (بالدرجات) من النقطة (lat, lon).
    """
    distances = np.sqrt((points['LAT'] - lat)**2 + (points['LON'] - lon)**2)
    nearby_points = points[distances <= max_distance]
    
    if 'elevation' in nearby_points.columns and not nearby_points.empty:
        valid_elevations = nearby_points['elevation'].dropna()
        if not valid_elevations.empty:
            # تحذير لو النقاط القريبة بعيدة
            max_dist = distances[distances <= max_distance].max()
            if max_dist > 0.2:  # 0.2 درجة ≈ 22 كم
                print(f"تحذير: أقرب نقاط لـ ({lat}, {lon}) بعيدة ({max_dist:.4f} درجة)")
            return valid_elevations.mean()
    
    return np.nan

# جلب الارتفاع للنقاط اللي لسة مالهاش ارتفاع
missing_elevations = unique_points[unique_points['elevation'].isna()]
if not missing_elevations.empty:
    print(f"جلب الارتفاعات لـ {len(missing_elevations)} نقاط...")
    elevations = []
    for index, row in tqdm(missing_elevations.iterrows(), total=len(missing_elevations), desc="جلب الارتفاعات"):
        elevation = get_elevation(row['LAT'], row['LON'])
        elevations.append(elevation)
        # تأخير صغير لتجنب الضغط على الـ API
        time.sleep(0.1)
    
    # تحديث الارتفاعات في unique_points
    unique_points.loc[unique_points['elevation'].isna(), 'elevation'] = elevations
else:
    print("جميع النقاط لها ارتفعات بالفعل في ملف التخزين المؤقت")

# حفظ الارتفاعات في ملف التخزين المؤقت
unique_points[['LAT', 'LON', 'elevation']].to_csv(cache_file, index=False)
print(f"تم حفظ الارتفاعات في {cache_file}")

# دمج الارتفاعات مع الداتا الأصلية
data_with_elevation = data.merge(
    unique_points[['LAT', 'LON', 'elevation']],
    on=['LAT', 'LON'],
    how='left'
)

# التحقق من وجود قيم فاضية
if data_with_elevation['elevation'].isna().any():
    print("تحذير: يوجد قيم فاضية في عمود elevation. جاري محاولة ملئها...")
    for index, row in data_with_elevation[data_with_elevation['elevation'].isna()].iterrows():
        elevation = get_nearby_elevation(row['LAT'], row['LON'], unique_points)
        data_with_elevation.at[index, 'elevation'] = elevation

# حفظ الداتا في ملف جديد
data_with_elevation.to_csv('climate_data_with_elevation.csv', index=False)
print("تم إضافة عمود 'elevation' وحفظ الداتا في 'climate_data_with_elevation.csv'")
print(f"عدد القيم الفاضية في عمود elevation: {data_with_elevation['elevation'].isna().sum()}")

عدد النقاط الفريدة: 90
جلب الارتفاعات لـ 90 نقاط...


جلب الارتفاعات: 100%|██████████| 90/90 [00:57<00:00,  1.58it/s]


تم حفظ الارتفاعات في unique_points_elevations.csv
تم إضافة عمود 'elevation' وحفظ الداتا في 'climate_data_with_elevation.csv'
عدد القيم الفاضية في عمود elevation: 0
